# Laboratório - Agentes baseados em objetivos e Métodos de Busca II

## Ajustando ambiente

In [ ]:
%%capture

import sys
import pandas as pd

## Motivação

O estudo da **busca informada (ou heurística)** é fundamental na
Inteligência Artificial porque fornece aos agentes computacionais os
mecanismos necessários para resolver problemas em larga escala de
maneira muito mais eficiente do que as estratégias cegas, utilizando
dicas específicas do domínio para guiar a exploração do espaço de
estados por meio de estimativas heurísticas. Essa capacidade de
otimização é o que viabiliza o desenvolvimento de aplicações práticas
complexas do mundo real, como os serviços de mapas e GPS que calculam
rotas ótimas em malhas com dezenas de milhões de nós em questão de
milissegundos, ou a resolução de tarefas com espaços de estados
massivos, que rapidamente esgotariam a memória de algoritmos não
informados. A literatura apresenta diversos exemplos de métodos
informados projetados para diferentes necessidades de tempo e espaço,
como a **Busca em Feixe**, a **Busca A\* Ponderada**, o **IDA** e o
**SMA**. Entretanto, o domínio dessa área passa fundamentalmente pela
compreensão de dois algoritmos centrais: a **Busca Gulosa pela Melhor
Escolha** e a **Busca A\***. A busca gulosa atua de forma imediatista,
direcionando o algoritmo a expandir sempre o nó que aparenta estar mais
próximo do objetivo com base exclusivamente na heurística, o que a torna
uma abordagem rápida, porém suscetível a soluções subótimas (caminhos
mais caros e longos), uma vez que ignora totalmente o custo das ações
passadas. Em contrapartida, o **algoritmo A\*** se destaca como a
estratégia informada mais comum por corrigir essa falha: ele equilibra a
exploração ao somar o custo real exato do caminho já percorrido com a
estimativa do custo restante até a meta, sendo capaz de podar
sistematicamente ramos irrelevantes da árvore de busca e garantir uma
solução perfeitamente ótima e completa, desde que utilize uma heurística
admissível (otimista).

## Objetivos de Aprendizagem

- **Entender a diferença entre GBFS e A\*:** compreender por que a busca
  gulosa considera só a heurística, enquanto o A\* combina heurística e
  custo acumulado.
- **Interpretar `g(n)`, `h(n)` e `f(n)`:** entender o papel de cada
  termo na escolha do próximo caminho a expandir.
- **Observar o efeito da heurística:** como diferentes heurísticas mudam
  a ordem de expansão e o caminho encontrado.
- **Implementar os algoritmos:** diretamente sobre um grafo representado
  como dicionário, com a heurística como parâmetro explícito.
- **Comparar heurísticas:** avaliar o mesmo problema de busca sob
  heurísticas mais e menos informativas, incluindo uma heurística
  propositalmente enganosa.

## Implementação

### Representação do ambiente

Assim como no capítulo anterior, o ambiente é só um dicionário: cada
cidade aponta para suas vizinhas (com o custo de cada aresta).

In [ ]:
environment = {
    'Natal': {'Parnamirim': 1, 'Extremoz': 1, 'São Gonçalo do Amarante': 1, 'Macaíba': 1},
    'Parnamirim': {'Natal': 1, 'São José de Mipibu': 1, 'Macaíba': 1},
    'Extremoz': {'Natal': 1},
    'São Gonçalo do Amarante': {'Natal': 1, 'Macaíba': 1, 'Ceará-Mirim': 1},
    'Macaíba': {'Natal': 1, 'Parnamirim': 1, 'São Gonçalo do Amarante': 1, 'Ielmo Marinho': 1, 'Vera Cruz': 1},
    'Ceará-Mirim': {'São Gonçalo do Amarante': 1},
    'Ielmo Marinho': {'Macaíba': 1},
    'Vera Cruz': {'Macaíba': 1, 'Monte Alegre': 1},
    'Monte Alegre': {'Vera Cruz': 1, 'São José de Mipibu': 1},
    'São José de Mipibu': {'Parnamirim': 1, 'Goianinha': 1, 'Monte Alegre': 1},
    'Goianinha': {'São José de Mipibu': 1, 'Tibau do Sul': 1},
    'Tibau do Sul': {'Goianinha': 1, 'Pipa': 1},
    'Pipa': {'Tibau do Sul': 1}
}

### Funções auxiliares

Cada candidato na fronteira de busca é o **caminho percorrido até ali**
(uma lista de cidades), não um objeto `Node`. O estado atual é
`path[-1]`.

In [ ]:
def path_cost(graph, path):
    """Soma o custo das arestas percorridas em um caminho."""
    return sum(graph[a][b] for a, b in zip(path, path[1:]))

Diferente do capítulo 1, aqui as arestas têm pesos diferentes (ver
`environment_with_distance` mais adiante), então `path_cost` já calcula
o $g(n)$ real — o custo acumulado do caminho — que os algoritmos abaixo
usam diretamente.

## Exemplo prático

Agora vamos aplicar essas funções em um cenário concreto, com um estado
inicial `Natal` e um objetivo `Pipa`.

In [ ]:
graph = environment
estado_inicial = 'Natal'
objetivo = 'Pipa'

print('Estado inicial:', estado_inicial)
print('Objetivo:', objetivo)
print('Ações possíveis em Natal:', list(graph[estado_inicial].keys()))

### Definindo heurística

Aqui iremos definir o dicionário relacionado à heurística da distância
em linha reta (informalmente associada à ideia da distância do ‘voo do
pássaro’). Em síntese, a heurística da distância em linha reta
($h_{SLD}$) se baseia em estimar o custo do caminho mais barato do
estado atual até o objetivo calculando a distância física direta
(geométrica) entre os dois pontos no mapa.

In [ ]:
heuristics = {
    "Ceará-Mirim": 76,
    "Extremoz": 64,
    "Goianinha": 18,
    "Ielmo Marinho": 71,
    "Macaíba": 52,
    "Monte Alegre": 35,
    "Natal": 45,
    "Parnamirim": 41,
    "Pipa": 0,
    "São Gonçalo do Amarante": 57,
    "São José de Mipibu": 27,
    "Tibau do Sul": 6,
    "Vera Cruz": 47
}

A heurística é só isso: um dicionário. Diferente da versão anterior
deste notebook, não vamos empacotá-la numa função/closure — cada
algoritmo de busca informada abaixo recebe `heuristics` diretamente como
parâmetro, então fica sempre explícito qual heurística está em uso em
cada chamada (sem depender de uma variável global reatribuída ao longo
do notebook).

### Relembrando a busca em profundidade

Para entender a implementação da Busca Gulosa pela Melhor Escolha
(*Greedy Best-First Search* ou GBFS), vale relembrar a busca em
profundidade (DFS) do capítulo anterior. Embora o algoritmo de DFS seja
um método de busca **não informada**, seu funcionamento serve como
fundamento essencial para a compreensão de certos métodos de busca
informada, sobretudo devido à sua mecânica de “mergulhar” profundamente
explorando um único caminho por vez e ao seu uso **extremamente
econômico de memória**.

In [ ]:
def depth_first_search(graph, start, goal, return_expansion_order=False):
    """Busca em profundidade: expande o caminho mais recente primeiro (pilha)."""
    frontier = [[start]]
    visited = set()
    expansion_order = []

    while frontier:
        path = frontier.pop()
        current = path[-1]
        expansion_order.append(current)

        if current == goal:
            return (path, expansion_order) if return_expansion_order else path

        if current in visited:
            continue
        visited.add(current)

        for neighbor in graph.get(current, {}):
            if neighbor not in visited:
                frontier.append(path + [neighbor])

    return (None, expansion_order) if return_expansion_order else None

No entanto, as diferenças essenciais entre a DFS e a GBFS estão
vinculadas ao critério de escolha direcionado pela incorporação da
heurística como método de priorização. A heurística funciona como
critério para seleção do nó que será removido da fronteira no caso da
Busca Gulosa pela Melhor Escolha (GBFS), onde uma função $h(n)$ estima o
custo do estado atual até o estado objetivo. O algoritmo GBFS utiliza
uma fila de prioridade para avaliar esses nós, expandindo primeiro
aquele que aparenta estar mais próximo da meta, agindo de forma
imediatista e ignorando o custo das ações já realizadas. Em
contrapartida, a Busca em Profundidade (DFS) é um método de busca não
informada, ou seja, ela não recebe nenhuma estimativa ou pista sobre a
localização do objetivo.

### Greedy Best-First Search

A busca gananciosa pela melhor escolha (*Greedy best-first search*) é
uma estratégia de busca informada na qual o nó que aparenta estar mais
próximo do estado objetivo é expandido primeiro. O algoritmo geralmente
utiliza uma estrutura de dados do tipo fila de prioridade, o que garante
que os nós gerados sejam ordenados com base exclusivamente em uma
estimativa heurística ($f(n) = h(n)$), permitindo que os nós com os
menores valores (mais promissores) sejam posicionados no topo da fila e
expandidos antes dos demais. Por focar apenas no benefício imediato de
se aproximar da meta a cada iteração e ignorar completamente o custo das
ações passadas, a busca gananciosa não é ótima (podendo gerar soluções
mais caras) e é incompleta em espaços de estados infinitos, embora
costume guiar a exploração de forma veloz e eficiente na maioria dos
casos práticos.

A ideia é a mesma da DFS (fronteira tratada como lista, teste de
objetivo ao retirar um caminho), com uma única mudança: em vez de tirar
o último caminho adicionado (`pop()`), ordenamos a fronteira pela
heurística do estado atual e tiramos sempre o mais promissor (`pop(0)`
depois de ordenar).

In [ ]:
def greedy_best_first_search(graph, start, goal, heuristics, return_expansion_order=False):
    """Busca gulosa: expande sempre o caminho cujo estado atual parece mais perto do objetivo (h(n))."""
    frontier = [[start]]
    visited = set()
    expansion_order = []

    while frontier:
        frontier.sort(key=lambda path: (heuristics[path[-1]], path[-1]))
        path = frontier.pop(0)
        current = path[-1]

        if current in visited:
            continue
        visited.add(current)
        expansion_order.append(current)

        if current == goal:
            return (path, expansion_order) if return_expansion_order else path

        for neighbor in graph.get(current, {}):
            if neighbor not in visited:
                frontier.append(path + [neighbor])

    return (None, expansion_order) if return_expansion_order else None

In [ ]:
resultado_gbfs, expansoes_gbfs = greedy_best_first_search(graph, estado_inicial, objetivo, heuristics, return_expansion_order=True)
print('Caminho encontrado (GBFS):', resultado_gbfs)
print('Ordem de expansão GBFS:', expansoes_gbfs)

### A-star Search

A busca A\* (A-estrela) é a estratégia de busca informada mais comum, na
qual o nó selecionado para expansão é aquele que apresenta o menor custo
total estimado do início até o objetivo. O algoritmo geralmente utiliza
uma estrutura de dados do tipo fila de prioridade, o que garante que os
nós gerados sejam ordenados pela função de avaliação
$f(n) = g(n) + h(n)$, que combina o custo exato do caminho percorrido
até o momento ($g(n)$) com a estimativa heurística do custo restante
($h(n)$). Por equilibrar o peso entre o custo das ações passadas e a
aproximação da meta. A busca A\* é completa e garante encontrar uma
solução perfeitamente ótima, desde que a heurística utilizada seja
admissível (ou seja, otimista, nunca superestimando o custo real para se
atingir o objetivo). O principal desafio desse algoritmo, no entanto,
reside no uso excessivo de espaço computacional, visto que ele precisa
armazenar todos os nós gerados na memória, o que pode esgotar os
recursos rapidamente em problemas de complexidade exponencial.

In [ ]:
environment_with_distance = {
    "Ceará-Mirim": {"Ielmo Marinho": 26, "São Gonçalo do Amarante": 21},
    "Extremoz": {"Natal": 25, "São Gonçalo do Amarante": 17},
    "Goianinha": {"São José de Mipibu": 23, "Tibau do Sul": 19},
    "Ielmo Marinho": {"Ceará-Mirim": 26, "Macaíba": 28},
    "Macaíba": {"Ielmo Marinho": 28, "Natal": 21, "Parnamirim": 16, "São Gonçalo do Amarante": 8, "Vera Cruz": 25},
    "Monte Alegre": {"São José de Mipibu": 12, "Vera Cruz": 13},
    "Natal": {"Extremoz": 25, "Macaíba": 21, "Parnamirim": 13, "São Gonçalo do Amarante": 22},
    "Parnamirim": {"Macaíba": 16, "Natal": 13, "São José de Mipibu": 19},
    "Pipa": {"Tibau do Sul": 9},
    "São Gonçalo do Amarante": {"Ceará-Mirim": 21, "Extremoz": 17, "Macaíba": 8, "Natal": 22},
    "São José de Mipibu": {"Goianinha": 23, "Monte Alegre": 12, "Parnamirim": 19},
    "Tibau do Sul": {"Goianinha": 19, "Pipa": 9},
    "Vera Cruz": {"Macaíba": 25, "Monte Alegre": 13}
}

graph = environment_with_distance

print('Estado inicial:', estado_inicial)
print('Objetivo:', objetivo)
print('Ações possíveis em Natal:', list(graph[estado_inicial].keys()))

A distância entre as cidades está representada nos valores do dicionário
interno, que representa as arestas do grafo — é a partir dele que
`path_cost` calcula $g(n)$ para cada caminho.

In [ ]:
def astar_search(graph, start, goal, heuristics, return_expansion_order=False):
    """Busca A*: expande sempre o caminho de menor custo total estimado (g(n) + h(n))."""
    frontier = [[start]]
    visited = set()
    expansion_order = []
    trace = []

    while frontier:
        frontier.sort(key=lambda path: (path_cost(graph, path) + heuristics[path[-1]], path[-1]))
        path = frontier.pop(0)
        current = path[-1]

        if current in visited:
            continue
        visited.add(current)
        expansion_order.append(current)

        g = path_cost(graph, path)
        trace.append({'estado': current, 'g(n)': g, 'h(n)': heuristics[current], 'f(n)': g + heuristics[current]})

        if current == goal:
            print(f"{'Estado':<24} {'g(n)':>6} {'h(n)':>6} {'f(n)':>6}")
            print('-' * 46)
            for row in trace:
                print(f"{row['estado']:<24} {row['g(n)']:>6} {row['h(n)']:>6} {row['f(n)']:>6}")
            return (path, expansion_order) if return_expansion_order else path

        for neighbor in graph.get(current, {}):
            if neighbor not in visited:
                frontier.append(path + [neighbor])

    return (None, expansion_order) if return_expansion_order else None

In [ ]:
resultado_astar, expansoes_astar = astar_search(graph, estado_inicial, objetivo, heuristics, return_expansion_order=True)

print('Caminho encontrado (A*):', resultado_astar)
print('Ordem de expansão A*:', expansoes_astar)

### Ajustando heurística

In [ ]:
new_heuristics = {
    'Natal': 74,
    'Parnamirim': 61,
    'Extremoz': 85,
    'São Gonçalo do Amarante': 84,
    'Macaíba': 71,
    'Ceará-Mirim': 99,
    'Ielmo Marinho': 85,
    'Vera Cruz': 57,
    'Monte Alegre': 48,
    'São José de Mipibu': 44,
    'Goianinha': 24,
    'Tibau do Sul': 7,
    'Pipa': 0,
}

In [ ]:
data = []
for city in sorted(heuristics.keys()):
    old_h = heuristics.get(city, 0)  # Use 0 as default if not found
    new_h = new_heuristics.get(city, 0) # Use 0 as default if not found
    difference = new_h - old_h
    data.append([city, old_h, new_h, difference])

df_heuristics = pd.DataFrame(data, columns=['Cidade', 'Heurística Antiga', 'Nova Heurística', 'Diferença'])
display(df_heuristics)

Explicação

Ao ajustar os valores da heurística, agora nos aproximamos de uma lógica
que representa de maneira mais fidedigna a **distância geográfica real
(ou o custo verdadeiro) das rotas até o estado objetivo**.

Como a literatura de Inteligência Artificial estabelece, **é geralmente
preferível utilizar uma função heurística com valores mais altos**,
desde que ela continue sendo admissível e consistente (ou seja, nunca
superestimando o custo real da viagem). Ao refinar esses valores para
que fiquem mais precisos e próximos da realidade, os contornos da busca
do algoritmo A\* deixam de se espalhar em todas as direções e passam a
se focar de forma muito mais estreita e direcionada para a meta.

Na prática, isso significa que cidades irrelevantes ou que representam
“andar para trás” receberão rapidamente estimativas de custo total
($f(n)$) superiores ao custo do caminho ótimo real. Isso permite que o
algoritmo **pode (elimine) esses caminhos desnecessários da árvore de
exploração sem precisar sequer visitá-los**. O resultado direto de uma
heurística mais fidedigna é um ganho substancial de eficiência, pois o
algoritmo expande uma quantidade significativamente menor de nós e
resolve o problema consumindo menos tempo e memória.

In [ ]:
resultado_astar, expansoes_astar = astar_search(graph, estado_inicial, objetivo, new_heuristics, return_expansion_order=True)
print('Caminho encontrado (A*):', resultado_astar)
print('Ordem de expansão A*:', expansoes_astar)

## Então, o Greedy é melhor?

Para evidenciar as fraquezas do Greedy Best-First Search, vamos usar uma
heurística propositalmente enganosa. Ela faz alguns estados parecerem
muito promissores localmente, mesmo quando isso não produz o melhor
caminho global.

In [ ]:
misleading_heuristics = {
    'Natal': 40,
    'Parnamirim': 35,
    'Extremoz': 20,
    'São Gonçalo do Amarante': 12,
    'Macaíba': 10,
    'Ceará-Mirim': 8,
    'Ielmo Marinho': 7,
    'Vera Cruz': 9,
    'Monte Alegre': 11,
    'São José de Mipibu': 25,
    'Goianinha': 18,
    'Tibau do Sul': 6,
    'Pipa': 0,
}

print('Heurística enganosa:')
for cidade, valor in misleading_heuristics.items():
    print(cidade, '-> h =', valor)

In [ ]:
resultado_greedy_ruim, expansoes_greedy_ruim = greedy_best_first_search(graph, estado_inicial, objetivo, misleading_heuristics, return_expansion_order=True)
resultado_astar_ruim, expansoes_astar_ruim = astar_search(graph, estado_inicial, objetivo, misleading_heuristics, return_expansion_order=True)

In [ ]:
print('Greedy com heurística enganosa:', resultado_greedy_ruim)
print('Ordem de expansão Greedy:', expansoes_greedy_ruim)
print()
print('A* com a mesma heurística:', resultado_astar_ruim)
print('Ordem de expansão A*:', expansoes_astar_ruim)

In [ ]:
acoes_astar = resultado_astar[1:] if resultado_astar else []

print('Agente com A*')
print('Ações:', acoes_astar)
print('Trajetória:', resultado_astar)

## Desafio

A busca de satisfação (*satisficing search*) é uma abordagem que abre
mão de encontrar a solução perfeitamente ótima em troca de explorar
significativamente menos nós, aceitando uma solução subótima que seja
“boa o suficiente” para economizar tempo e espaço computacional. O
principal exemplo dessa estratégia é o algoritmo **A\* ponderado**
(*weighted A\**), que permite o uso de heurísticas inadmissíveis (que
superestimam o custo real) ao aplicar um multiplicador $W > 1$ à
estimativa heurística, resultando na função de avaliação modificada
$f(n) = g(n) + W \times h(n)$. Ao dar um peso maior à distância
restante, o algoritmo assume um comportamento “um pouco guloso”, focando
o contorno da busca de forma mais agressiva e direta em direção ao
objetivo, sem ignorar totalmente o custo das ações passadas. Embora essa
inclinação sacrifique a otimização matemática garantida pelo A\*
tradicional — uma vez que a rota encontrada terá um custo que varia
entre o valor ótimo $C^*$ e $W \times C^*$ —, na prática o A\* ponderado
explora uma quantidade muito menor de estados e encontra soluções
extremamente rápidas com custos bem próximos do ideal.

Implemente uma variação do algoritmo **A\* Ponderado** para resolver o
problema de busca de **Natal** até **Pipa**, utilizando o ambiente
`environment_with_distance`. No A\* ponderado, a função de avaliação é
definida como:

$$f(n) = g(n) + w \cdot h(n)$$

Onde $w \ge 1$ representa o peso aplicado à heurística.

### Diretrizes de Implementação

1.  **Compatibilidade:** Utilize a mesma representação de grafo e
    caminho do `astar_search` implementado acima (caminho como lista,
    `path_cost` para $g(n)$).
2.  **Assinatura da Função:** A função deve aceitar os seguintes
    parâmetros:
    - `graph`, `start`, `goal`: mesmos parâmetros de `astar_search`.
    - `heuristics`: dicionário de heurística.
    - `w`: o peso da heurística (parâmetro de ponderação).
    - `return_expansion_order`: Booleano (padrão `False`) que define o
      formato do retorno.
3.  **Retorno:** Caso `return_expansion_order=True`, a função deve
    retornar uma tupla contendo o **caminho solução** e a **lista com a
    ordem de expansão**.
4.  **Casos de Teste:** Avalie o desempenho do algoritmo utilizando os
    seguintes pesos:
    - $w = 1.0$
    - $w = 1.5$
    - $w = 2.0$

In [ ]:
def weighted_astar_search(graph, start, goal, heuristics, w=1.0, return_expansion_order=False):
    """Busca A* ponderada: expande sempre o caminho de menor g(n) + w * h(n)."""
    pass

Dicas

Uma implementação possível de `weighted_astar_search`, adaptando
`astar_search` para ponderar a heurística por `w`:

``` python
def weighted_astar_search(graph, start, goal, heuristics, w=1.0, return_expansion_order=False):
    frontier = [[start]]
    visited = set()
    expansion_order = []

    while frontier:
        frontier.sort(key=lambda path: (path_cost(graph, path) + w * heuristics[path[-1]], path[-1]))
        path = frontier.pop(0)
        current = path[-1]

        if current in visited:
            continue
        visited.add(current)
        expansion_order.append(current)

        if current == goal:
            return (path, expansion_order) if return_expansion_order else path

        for neighbor in graph.get(current, {}):
            if neighbor not in visited:
                frontier.append(path + [neighbor])

    return (None, expansion_order) if return_expansion_order else None
```

## Perguntas para reflexão

1.  Quando `w = 1`, o A\* ponderado se comporta exatamente como o A\*
    tradicional. Por que isso decorre diretamente da fórmula
    $f(n) = g(n) + w \cdot h(n)$, e o que isso revela sobre o A\*
    clássico ser apenas um caso particular de uma família mais ampla de
    buscas?
2.  Por que aumentar `w` tende a reduzir o número de nós expandidos? Que
    trade-off está sendo feito entre velocidade da busca e qualidade
    (custo) da solução encontrada?
3.  A garantia de encontrar o caminho ótimo se perde à medida que `w`
    cresce. Explique por que isso acontece — o que muda na relação entre
    $f(n)$ e o custo real restante — e descreva um cenário (real ou
    hipotético) em que essa perda de otimalidade levaria a uma escolha
    claramente pior.
4.  Em que valor ou faixa de `w` o A\* ponderado passa a se comportar
    essencialmente como a Greedy Best-First Search? O que isso sugere
    sobre a Greedy ser, na prática, um caso extremo do A\* ponderado, em
    que o custo acumulado deixa de influenciar a decisão?

## Key takeaways

Greedy Best-First Search decide com base apenas na heurística, enquanto
o A\* combina custo acumulado e estimativa restante. A qualidade da
heurística afeta diretamente a eficiência da busca: uma heurística mais
informativa tende a reduzir expansões desnecessárias e a direcionar
melhor o algoritmo até o objetivo.

## Referencias

1.  Russell, S. & Norvig, P. (2010). Artificial Intelligence: A Modern
    Approach. Prentice Hall.